# Chapter 7: Using Examples (Few Shot Prompting)

**Technique:** Few shot prompting

**Contents**
* [Lesson: Showing, Not Just Telling](#lesson)
* [Exercises](#exercises)
* Example Playground

In [ ]:
import { getCompletion, MODEL, client } from "../lib/helpers.js";

## Lesson: Showing, Not Just Telling

The fastest way to steer Claude's output format and classification vocabulary is to show it a handful of worked examples before you ask your real question. This is called **few shot prompting**.

### Why examples beat descriptions

Describing output format in prose ("respond with exactly one word, lowercase, no punctuation") works, but it forces Claude to infer the mapping from your words. Examples give it the mapping directly:

```
User:   git commit -m "fix null deref in auth middleware"
Label:  fix

User:   git commit -m "add dark-mode toggle to settings panel"
Label:  feat

User:   git commit -m "bump lodash from 4.17.19 to 4.17.21"
Label:  chore

User:   git commit -m "remove deprecated /v1/users endpoint"
Label:
```

Claude sees the pattern (short lowercase conventional commit label, no trailing text) and applies it to the new input.

### How few shot examples work in the message

In a plain user message the examples are just text. You write them as if you're showing a colleague what "correct" looks like:

```js
const examples = [
  "git commit -m \"fix null deref in auth middleware\" -> fix",
  "git commit -m \"add dark-mode toggle to settings panel\" -> feat",
  "git commit -m \"bump lodash from 4.17.19 to 4.17.21\" -> chore",
].join("\n");

const COMMIT = 'git commit -m "remove deprecated /v1/users endpoint"';
const PROMPT = `Label each commit with its conventional-commit type.
Use only: feat / fix / chore / docs / refactor / test / style

Examples:
${examples}

Now label this commit (respond with the label only):
${COMMIT}`;
```

### This is NOT assistant turn prefilling

Older tutorials showed "few shot" by injecting partial `assistant` turns into the messages array. On current Claude models (Sonnet 4.6+) that returns a **400 Bad Request**. The approach above places examples entirely in the user message, with no extra API parameters, and works on all models.

### How many examples?

* **1 example**: establishes a pattern but may not generalize well.
* **2 to 4 examples**: covers edge cases and ambiguous inputs; the sweet spot for most tasks.
* **5+ examples**: useful for tasks with many output classes or subtle distinctions.

Start small and add examples only when the model gets it wrong on real inputs.

In [ ]:
// Few shot: label a git commit with its conventional commit type
const examples = [
  'git commit -m "fix null deref in auth middleware" -> fix',
  'git commit -m "add dark-mode toggle to settings panel" -> feat',
  'git commit -m "bump lodash from 4.17.19 to 4.17.21" -> chore',
].join("\n");

const COMMIT = 'git commit -m "remove deprecated /v1/users endpoint"';

const PROMPT = `Label each commit with its conventional-commit type.
Use only: feat / fix / chore / docs / refactor / test / style

Examples:
${examples}

Now label this commit (respond with the label only):
${COMMIT}`;

console.log(await getCompletion(PROMPT));

## Exercises

### Exercise 7.1: Label a log line by severity

**Scenario**: a log ingestion pipeline needs to route each incoming log line to the right alert queue. The router reads the **last word** of Claude's reply as the severity label. Your job is to give Claude a few worked examples so it maps the new log line to exactly the right label.

**Task**: replace the scaffold prompt with a few shot prompt that includes 2 to 3 input→output examples, then asks Claude to label the final log line. The label must be the **last token** of Claude's reply. You can add a brief cue like "Label:" on the final line so Claude knows exactly where to put it.

Target log line: `"DB connection pool exhausted; requests timing out"`

**Success criteria**: the last token of Claude's response is `error`.

In [ ]:
import { grade } from "../lib/grading.js";

const LOG = "DB connection pool exhausted; requests timing out";
const PROMPT = "[Replace this text - give few-shot examples, then ask for the label of the final log line as the last word]\n" + LOG;

const response = await getCompletion(PROMPT);
const lastToken = response.trim().split(/\s+/).pop().toLowerCase().replace(/[^a-z]/g, "");
const gradeExercise = () => lastToken === "error";
grade(response, gradeExercise());

In [ ]:
import { exercise_7_1_hint } from "../hints.js";
console.log(exercise_7_1_hint);

## Common mistakes, stretch challenge, and reflection

**Common mistakes**

* **Inconsistent example formatting**: if some examples say "Label: warn" and others say just "warn", the model averages them and may add or drop the prefix unpredictably. Pick one format and apply it to every example.
* **Too few examples for a multiclass problem**: with four possible severity levels (debug / info / warn / error) one example covers only one class. Add at least one example per class you care about most.
* **Letting the label slip mid sentence**: if your examples end mid sentence ("...this is a warn level event"), Claude may do the same. Terminate each example with the bare label on its own line or after a clear separator like `->`.

**Stretch challenge**

Add a fourth log line that is genuinely ambiguous (for example, `"Retrying request (attempt 2 of 3)"`) and observe how your examples bias Claude's answer. Then swap in different examples that would tip it toward `warn` instead of `info`. How many examples does it take to flip the result?

**Reflect**: few shot prompting vs. structured output (`output_config.format`): when do you reach for each? Consider: few shot is fast to author and needs no schema; structured output guarantees a parseable shape and lets you add fields later without touching the prompt. For a production pipeline routing 10 k log lines per second, which would you choose and why?

## Example Playground

Use the cells below to experiment freely. Try different example sets, vary the number of shots, or combine few shot examples with an explicit system prompt role.

In [ ]:
// Experiment: classify log lines with a custom few shot set
const fewShot = [
  "Server started on port 8080 -> info",
  "Retrying DB connection (attempt 1 of 5) -> warn",
  "Unhandled exception in request handler: TypeError: Cannot read property 'id' of undefined -> error",
  "Cache miss for key user:42 -> debug",
].join("\n");

const LOG_LINES = [
  "Disk usage at 91% on /dev/sda1",
  "User login successful: user@example.com",
  "Payment gateway timeout after 30s",
];

for (const line of LOG_LINES) {
  const PROMPT = `Classify each log line as debug / info / warn / error.

Examples:
${fewShot}

Now classify this line (respond with the label only):
${line}`;

  const label = await getCompletion(PROMPT);
  console.log(`[${label.trim()}] ${line}`);
}